# 02 · Reproductive Feature Engineering
## NHANES 2017–March 2020 Women's CKM Phenotyping Project

---

**Author:** Alexandra Velez, OB-GYN (Colombia) | Data Science  
**Input:** `data/processed/nhanes_clean_sample.csv` — 1,603 women aged 20–44  
**Output:** `data/processed/reproductive_features.csv`  
**Last updated:** 2026

---

### Purpose of this notebook

This notebook engineers reproductive health features across four clinical 
domains — menstrual history, pregnancy history and adverse outcomes, surgical 
history, and hormone therapy — from the cleaned analytical sample produced in 
notebook 01.

The central clinical question driving feature construction is:

> Among reproductive-age women aged 20–44, do those with adverse pregnancy 
> outcomes already show distinct early cardiometabolic risk profiles that 
> are identifiable through unsupervised clustering?

This question was deliberately scoped to the 20–44 age range, driven by 
both a data limitation and a clinical opportunity. CDC suppresses pregnancy and hysterectomy data for women over 44 in 
the public use file due to disclosure risk, restricting the analytical sample 
to reproductive-age women only. However, this constraint aligns with a genuine 
clinical opportunity: women aged 20–44 represent the intervention window before 
overt cardiovascular disease manifests. Their cardiometabolic biomarkers are 
already measurable and may already be diverging based on reproductive history — 
even if cardiovascular outcomes have not yet had time to develop. The analysis 
therefore focuses on early cardiometabolic dysregulation rather than late-stage 
outcomes, which is both the honest scope of what this data supports and the 
clinically meaningful question for preventive medicine.

Features engineered here will be used to identify early cardiometabolic risk 
profiles in notebook 05. This analysis makes no causal claims — it identifies 
associations between reproductive history and concurrent metabolic biomarker 
profiles. Longitudinal data with outcome follow-up would be required to 
establish causal relationships between adverse pregnancy outcomes and 
cardiovascular disease.

---

### Notebook inputs and outputs

| File | Description |
|---|---|
| `data/processed/nhanes_clean_sample.csv` | Analytical sample from notebook 01 |
| `data/processed/reproductive_features.csv` | Engineered feature matrix — input for notebook 03 |

---

### Engineering decisions carried forward from notebook 01

The following decisions made during exploration directly affect feature 
construction in this notebook:

| Variable | Issue | Action |
|---|---|---|
| RHD018 | Age in months, not years | Divide by 12 |
| RHQ010 | Code 0 = pre-menarche | Handle explicitly — not missing |
| RHQ160 | Code 11 = 11 or more pregnancies | Top-coded — treat as 11 |
| RHD167 | Code 5 = 5 or more deliveries | Top-coded — treat as 5 |
| RHQ162 | Code 3 = borderline GDM | Treat as Yes — conservative clinical decision |
| RHQ020 | Ordinal fallback for menarche age | Use midpoint imputation |
| RHQ070 | Ordinal fallback for menopause age | Use midpoint imputation |
| RHQ542A–D | Multi-select hormone therapy forms | Build composite HRT_Type |

## Section 1 · Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from pathlib import Path
import warnings
import os

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH  = Path('../data/processed/nhanes_clean_sample.csv')
OUTPUT_PATH = Path('../data/processed/reproductive_features.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')
print(f'  Input:  {INPUT_PATH}')
print(f'  Output: {OUTPUT_PATH}')

Setup complete.
  Input:  ../data/processed/nhanes_clean_sample.csv
  Output: ../data/processed/reproductive_features.csv


---
## Section 2 · Load Analytical Sample

The analytical sample was fully validated and documented in notebook 01. 
The three assertions below confirm the CSV handoff was clean — shape, 
index, and age range.

In [2]:
# ── Load analytical sample ─────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH, index_col='SEQN')

# Light validation — confirms handoff from notebook 01 was clean
assert df.shape == (1603, 42), \
    f'Unexpected shape {df.shape} — expected (1603, 42)'
assert df.index.name == 'SEQN', \
    'SEQN is not the index — check CSV export from notebook 01'
assert df['RIDAGEYR'].between(20, 44).all(), \
    'Age restriction violated — participants outside 20–44 present'

print(f'✓ Analytical sample loaded: {df.shape}')
print(f'  Age range:  {df["RIDAGEYR"].min():.0f}–{df["RIDAGEYR"].max():.0f} years')
print(f'  SEQN index: confirmed')
print(f'  Columns:    {df.shape[1]} '
      f'(30 P_RHQ variables + 12 P_DEMO variables)')

✓ Analytical sample loaded: (1603, 42)
  Age range:  20–44 years
  SEQN index: confirmed
  Columns:    42 (30 P_RHQ variables + 12 P_DEMO variables)


---
## Section 3 · Binary Variable Recoding

All binary variables in the analytical sample retain their raw NHANES coding 
(1=Yes, 2=No) from notebook 01 — recoding was deliberately deferred to this 
notebook to preserve the boundary between exploration and engineering.

This section applies permanent recoding across all binary variables using 
`recode_yes_no()`, converting 1→1.0 and 2→0.0 with NaN preserved. One 
variable requires special handling before recoding:

**RHQ162 — Gestational diabetes:**
Code 3 (borderline GDM) is treated as Yes (1.0) — a conservative clinical 
decision. Borderline GDM carries similar metabolic implications to confirmed 
GDM and represents the same underlying pathophysiology of impaired glucose 
tolerance under metabolic stress. Collapsing it into the Yes category is 
the clinically appropriate choice for a cardiometabolic risk analysis.


In [3]:
# ── Binary variable recoding ───────────────────────────────────────────────
# Permanently recode all binary variables from NHANES coding (1=Yes, 2=No)
# to analytical coding (1.0=Yes, 0.0=No, NaN=missing)
#
# This is a permanent transformation — df is modified in place
# All downstream feature engineering uses the recoded values

def recode_yes_no(series):
    """Convert NHANES binary coding (1=Yes, 2=No) to (1.0=Yes, 0.0=No, NaN)."""
    return series.map({1.0: 1.0, 2.0: 0.0})

# Binary variables in P_RHQ
binary_vars = [
    'RHQ031',   # Regular periods in past 12 months
    'RHQ074',   # Tried ≥1 year to conceive without success
    'RHQ076',   # Seen doctor for inability to conceive
    'RHQ078',   # Ever treated for PID
    'RHQ131',   # Ever been pregnant
    'RHD143',   # Currently pregnant
    'RHQ172',   # Any baby weighed 9 lbs or more
    'RHQ200',   # Currently breastfeeding
    'RHD280',   # Had a hysterectomy
    'RHQ305',   # Had both ovaries removed
    'RHQ540',   # Ever used female hormones
    'RHQ554',   # Ever used estrogen-only pills
    'RHQ570',   # Ever used estrogen/progestin combo pills
]

# ── Special handling: RHQ162 borderline GDM → Yes ─────────────────────────
# Code 3 (borderline GDM) treated as Yes before recoding
# Clinical rationale: borderline GDM reflects same underlying impaired
# glucose tolerance as confirmed GDM — conservative classification
n_borderline = (df['RHQ162'] == 3).sum()
df['RHQ162'] = df['RHQ162'].replace(3, 1)
print(f'RHQ162 borderline GDM (code 3) → Yes: {n_borderline:,} participants reclassified')

# Add RHQ162 to binary vars after special handling
binary_vars.append('RHQ162')

# ── Apply recoding ─────────────────────────────────────────────────────────
print(f'\nApplying recode_yes_no() to {len(binary_vars)} binary variables...')
for var in binary_vars:
    df[var] = recode_yes_no(df[var])

# ── Validate ───────────────────────────────────────────────────────────────
print('\nValidation — no raw NHANES codes (1.0/2.0) should remain:')
issues = []
for var in binary_vars:
    if (df[var] == 2.0).any():
        issues.append(var)

if issues:
    print(f'  ⚠️  Raw coding still present in: {issues}')
else:
    print(f'  ✓ All {len(binary_vars)} variables confirmed recoded')
    print(f'  ✓ Only 1.0, 0.0, and NaN present in all binary variables')

# ── Summary ────────────────────────────────────────────────────────────────
print(f'\nBinary variable recoding summary:')
print(f'  {"Variable":<12} {"Yes (1.0)":>10} {"No (0.0)":>10} {"NaN":>8} {"% Yes":>8}')
print(f'  {"─"*12} {"─"*10} {"─"*10} {"─"*8} {"─"*8}')
for var in binary_vars:
    n_yes  = (df[var] == 1.0).sum()
    n_no   = (df[var] == 0.0).sum()
    n_nan  = df[var].isna().sum()
    pct    = n_yes / (n_yes + n_no) * 100 if (n_yes + n_no) > 0 else 0
    print(f'  {var:<12} {n_yes:>10,} {n_no:>10,} {n_nan:>8,} {pct:>7.1f}%')

RHQ162 borderline GDM (code 3) → Yes: 6 participants reclassified

Applying recode_yes_no() to 14 binary variables...

Validation — no raw NHANES codes (1.0/2.0) should remain:
  ✓ All 14 variables confirmed recoded
  ✓ Only 1.0, 0.0, and NaN present in all binary variables

Binary variable recoding summary:
  Variable      Yes (1.0)   No (0.0)      NaN    % Yes
  ──────────── ────────── ────────── ──────── ────────
  RHQ031            1,435        166        2    89.6%
  RHQ074              191      1,409        3    11.9%
  RHQ076              123      1,478        2     7.7%
  RHQ078               90      1,500       13     5.7%
  RHQ131            1,150        451        2    71.8%
  RHD143               67      1,009      527     6.2%
  RHQ172              144        917      542    13.6%
  RHQ200               58        177    1,368    24.7%
  RHD280               61      1,523       19     3.9%
  RHQ305               15      1,565       23     0.9%
  RHQ540               66     

In [4]:
# Verify hormone therapy routing
n_hrt_users = (df['RHQ540'] == 1.0).sum()
n_estrogen_only = (df['RHQ554'] == 1.0).sum()
n_combo = (df['RHQ570'] == 1.0).sum()

print(f'Ever used hormones (RHQ540=Yes):     {n_hrt_users:,}')
print(f'Among hormone users:')
print(f'  Estrogen-only pills (RHQ554=Yes):  {n_estrogen_only:,} '
      f'({n_estrogen_only/n_hrt_users*100:.1f}% of users)')
print(f'  Combo pills (RHQ570=Yes):          {n_combo:,} '
      f'({n_combo/n_hrt_users*100:.1f}% of users)')
print(f'\nAs % of full analytical sample (n=1,603):')
print(f'  Estrogen-only pills:               '
      f'{n_estrogen_only/len(df)*100:.1f}%')
print(f'  Combo pills:                       '
      f'{n_combo/len(df)*100:.1f}%')

Ever used hormones (RHQ540=Yes):     66
Among hormone users:
  Estrogen-only pills (RHQ554=Yes):  11 (16.7% of users)
  Combo pills (RHQ570=Yes):          6 (9.1% of users)

As % of full analytical sample (n=1,603):
  Estrogen-only pills:               0.7%
  Combo pills:                       0.4%


### Recoding complete — key observations

All 14 binary variables successfully recoded from NHANES coding (1=Yes, 2=No) 
to analytical coding (1.0=Yes, 0.0=No, NaN=missing). Six women with borderline 
GDM (RHQ162 code 3) were reclassified as Yes before recoding — a conservative 
clinical decision reflecting that borderline GDM represents the same underlying 
impaired glucose tolerance as confirmed GDM.

Several patterns in the recoding summary are worth noting before feature 
engineering begins:

**Structural missingness from skip logic:**
- RHD143 (currently pregnant): 527 NaN — only administered to women who 
  answered Yes to RHQ131 (ever pregnant). Women who have never been pregnant 
  were never asked this question — their NaN is structural, not missing data.
- RHQ172 (macrosomia): 542 NaN — same gate as RHD143
- RHQ200 (breastfeeding): 1,368 NaN — asked only of women who delivered 
  in the past 2 years, extremely sparse

**Hormone therapy variables:**
- RHQ540 (ever used hormones): only 66 Yes (4.1%) — expected for 
  reproductive-age women 20–44
- RHQ554 (Use hormone pills w/estrogen only) and RHQ570 (Used estrogen/progestin combo pills): sparse by design, gated behind RHQ540
- These will be combined into a single composite feature in Section 7

**Key exposure variables:**
- RHQ131 (ever pregnant): 71.8% Yes — primary stratification variable
- RHQ162 (GDM): 11.8% Yes among those asked — primary APO exposure
- RHQ172 (macrosomia): 13.6% Yes among those asked

---
## Section 4 · Domain 1: Menstrual History Features

Menstrual history variables capture the reproductive lifespan — from menarche 
to menopause — and provide context for understanding estrogen exposure duration, 
cycle regularity, and reproductive aging. These features are clinically relevant 
to CKM risk because estrogen has protective effects on cardiovascular and 
metabolic function. Earlier menopause, shorter reproductive span, and irregular 
cycles are all associated with adverse cardiometabolic trajectories.

Four features are derived in this section:

- **Age_Menarche** — age at first menstrual period in years, derived primarily 
  from RHQ010 (exact age in years, asked of all females aged 12–150). For women 
  who refused or did not know their exact age, ordinal midpoint imputation is 
  applied using RHQ020 (age range at first menstrual period), which provides a 
  range category rather than an exact age. The midpoint of each range is assigned 
  as the best available estimate.

- **Irregular_Periods** — binary flag for absence of regular periods in the 
  past 12 months, derived from RHQ031. A value of 1.0 indicates irregular or 
  absent periods — the clinically meaningful direction for CKM risk.

- **Menopause_Status** — categorical variable distinguishing premenopausal, 
  surgically menopausal, and naturally menopausal women. Derived from the 
  combination of RHQ031 (regular periods), RHD043 (reason for no periods), 
  RHD280 (hysterectomy), and RHQ305 (oophorectomy).

- **Age_Menopause** — age at menopause in years, derived from RHQ060 (exact 
  age) with ordinal midpoint imputation from RHQ070 where needed. Calculated 
  only for postmenopausal women.

- **Reproductive_Span** — continuous variable capturing total years of 
  endogenous estrogen exposure (Age_Menopause − Age_Menarche). Calculated 
  only for postmenopausal women with both ages known. Women aged 20–44 who 
  report natural menopause represent premature ovarian insufficiency — a 
  clinically recognized condition associated with accelerated cardiometabolic 
  risk.

### Engineering decisions for this domain

**RHQ010 = 0 (pre-menarche):**
One woman in the analytical sample has RHQ010 = 0, indicating menarche had 
not yet started at time of interview at age 27. This is consistent with 
primary amenorrhea. This woman will have NaN for all menstrual features and 
contributes to non-menstrual analyses only.

**RHQ020 ordinal midpoint values:**

| Code | Range | Midpoint assigned |
|---|---|---|
| 1 | Younger than 9 years | 8.5 |
| 2 | 9–10 years | 9.5 |
| 3 | 11–12 years | 11.5 |
| 4 | 13–14 years | 13.5 |
| 5 | 15–16 years | 15.5 |
| 6 | 17–19 years | 18.0 |
| 7 | 20 years or older | 20.0 |

**Menopause_Status derivation logic:**
Menopause status cannot be read from a single variable — it must be inferred 
from multiple variables. The derivation priority is:

1. Surgical menopause — hysterectomy (RHD280=1) OR bilateral oophorectomy 
   (RHQ305=1) → Surgical_Menopause
2. Natural menopause — no regular periods (RHQ031=0) AND reason is menopause 
   (RHD043=2) AND no surgical history → Natural_Menopause
3. All others with valid menstrual data → Premenopausal

In [5]:
# ── Domain 1: Menstrual history features ──────────────────────────────────
# Features derived in this section:
#   Age_Menarche      — age at first menstrual period in years
#   Irregular_Periods — binary flag for unexplained menstrual irregularity
#   Menopause_Status  — premenopausal vs surgical menopause
#   Age_Menopause     — age at surgical menopause (surgical only)
#   Reproductive_Span — years of endogenous estrogen exposure (surgical only)

features = pd.DataFrame(index=df.index)

# ── Feature 1: Age_Menarche ────────────────────────────────────────────────
# Primary source: RHQ010 (exact age in years, asked 12–150)
# Fallback: RHQ020 ordinal midpoint imputation where RHQ010 is NaN
# Exclusion: RHQ010 == 0 (primary amenorrhea) → NaN

ordinal_midpoints_menarche = {
    1.0:  8.5,   # < 9 years
    2.0:  9.5,   # 9–10 years
    3.0: 11.5,   # 11–12 years
    4.0: 13.5,   # 13–14 years
    5.0: 15.5,   # 15–16 years
    6.0: 18.0,   # 17–19 years
    7.0: 20.0,   # ≥ 20 years
}

age_menarche     = df['RHQ010'].copy().astype(float)
premenarche_mask = age_menarche == 0.0
age_menarche[premenarche_mask] = np.nan

rhq020_imputed = df['RHQ020'].map(ordinal_midpoints_menarche)
n_imputed      = (age_menarche.isna() & rhq020_imputed.notna()).sum()
age_menarche   = age_menarche.fillna(rhq020_imputed)

features['Age_Menarche'] = age_menarche

print('Age_Menarche construction:')
print(f'  From RHQ010 (exact age):         '
      f'{(df["RHQ010"].notna() & (df["RHQ010"] > 0)).sum():,}')
print(f'  From RHQ020 (ordinal imputed):   {n_imputed:,}')
print(f'  Pre-menarche (RHQ010=0) → NaN:  {premenarche_mask.sum():,}')
print(f'  Total valid:                     '
      f'{features["Age_Menarche"].notna().sum():,}')
print(f'  Total NaN:                       '
      f'{features["Age_Menarche"].isna().sum():,}')

# ── Feature 2: Irregular_Periods ──────────────────────────────────────────
# RHQ031 = 0 (no regular periods in past 12 months)
# Refined using RHD043 (reason for no periods) to exclude known
# non-CKM-relevant causes:
#   Code 1 = Pregnant       → exclude
#   Code 2 = Breastfeeding  → exclude
#   Code 3 = Hysterectomy   → exclude (captured in Menopause_Status)
#   Code 7 = Menopause      → absent in 20–44 sample
#   NaN    = Unknown reason → retained conservatively
#
# Final flag captures unexplained menstrual irregularity only

irregular_base    = df['RHQ031'] == 0.0
exclude_irregular = (
    (df['RHD043'] == 1.0) |   # pregnant
    (df['RHD043'] == 2.0) |   # breastfeeding
    (df['RHD043'] == 3.0)     # hysterectomy
)

features['Irregular_Periods'] = np.where(
    df['RHQ031'].isna(), np.nan,
    np.where(irregular_base & ~exclude_irregular, 1.0, 0.0)
)

print(f'\nIrregular_Periods construction:')
print(f'  Raw No to RHQ031:              {irregular_base.sum():,}')
print(f'  Excluded — pregnant:           '
      f'{(irregular_base & (df["RHD043"] == 1.0)).sum():,}')
print(f'  Excluded — breastfeeding:      '
      f'{(irregular_base & (df["RHD043"] == 2.0)).sum():,}')
print(f'  Excluded — hysterectomy:       '
      f'{(irregular_base & (df["RHD043"] == 3.0)).sum():,}')
print(f'  Retained — unknown reason:     '
      f'{(irregular_base & df["RHD043"].isna()).sum():,}')
print(f'  Final Irregular_Periods = 1:   '
      f'{(features["Irregular_Periods"] == 1.0).sum():,}')

# ── Feature 3: Menopause_Status ───────────────────────────────────────────
# Two categories only — Natural_Menopause not identifiable in 20–44 sample
# RHD043 code 7 (menopause) is absent in this age group
#
# Surgical_Menopause: hysterectomy (RHD280=1) OR oophorectomy (RHQ305=1)
# Premenopausal: all others with valid menstrual data

menopause_status              = pd.Series('Premenopausal', index=df.index,
                                          dtype=object)
surgical_mask                 = (df['RHD280'] == 1.0) | (df['RHQ305'] == 1.0)
menopause_status[surgical_mask]    = 'Surgical_Menopause'
menopause_status[premenarche_mask] = np.nan

features['Menopause_Status'] = menopause_status

print(f'\nMenopause_Status construction:')
for status in ['Premenopausal', 'Surgical_Menopause']:
    n   = (features['Menopause_Status'] == status).sum()
    pct = n / len(features) * 100
    print(f'  {status:<22} {n:>5,}  ({pct:.1f}%)')
print(f'  {"NaN":<22} '
      f'{features["Menopause_Status"].isna().sum():>5,}')
print(f'\n  Note: Natural_Menopause not derivable — RHD043 code 7')
print(f'  (menopause) absent in women aged 20–44')

# ── Features 4 & 5: Age_Menopause and Reproductive_Span ───────────────────
# Restricted to Surgical_Menopause women only
# Age_Menopause: RHQ060 (exact age) with RHQ070 ordinal fallback
# Reproductive_Span: Age_Menopause - Age_Menarche

ordinal_midpoints_menopause = {
    1.0: 20.0,   # < 25 years
    2.0: 27.0,   # 25–29 years
    3.0: 32.0,   # 30–34 years
    4.0: 37.0,   # 35–39 years
    5.0: 42.0,   # 40–44 years
    6.0: 47.0,   # 45–49 years
    7.0: 52.0,   # 50–54 years
    8.0: 55.0,   # ≥ 55 years
}

# Postmenopausal mask — surgical only after correction
postmeno_mask  = features['Menopause_Status'] == 'Surgical_Menopause'

age_menopause  = df['RHQ060'].copy().astype(float)
rhq070_imputed = df['RHQ070'].map(ordinal_midpoints_menopause)
age_menopause  = age_menopause.fillna(rhq070_imputed)

# Restrict to surgical menopause only
features['Age_Menopause'] = np.nan
features.loc[postmeno_mask, 'Age_Menopause'] = age_menopause[postmeno_mask]

# Reproductive span
features['Reproductive_Span'] = np.nan
features.loc[postmeno_mask, 'Reproductive_Span'] = (
    features.loc[postmeno_mask, 'Age_Menopause'] -
    features.loc[postmeno_mask, 'Age_Menarche']
)

# ── Domain 1 summary ───────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print(f'DOMAIN 1 SUMMARY — Menstrual History Features')
print(f'{"═"*60}\n')

print(f'  {"Feature":<22} {"Valid":>6} {"NaN":>6}  {"Key statistic"}')
print(f'  {"─"*22} {"─"*6} {"─"*6}  {"─"*30}')

# Age_Menarche
v = features['Age_Menarche'].notna().sum()
print(f'  {"Age_Menarche":<22} {v:>6,} '
      f'{len(features)-v:>6,}  '
      f'Mean {features["Age_Menarche"].mean():.1f} yrs  '
      f'Range {features["Age_Menarche"].min():.0f}–'
      f'{features["Age_Menarche"].max():.0f}')

# Irregular_Periods
v = (features['Irregular_Periods'] == 1.0).sum()
print(f'  {"Irregular_Periods":<22} {v:>6,} '
      f'{features["Irregular_Periods"].isna().sum():>6,}  '
      f'{v/len(features)*100:.1f}% of analytical sample')

# Menopause_Status
v = features['Menopause_Status'].notna().sum()
s = (features['Menopause_Status'] == 'Surgical_Menopause').sum()
print(f'  {"Menopause_Status":<22} {v:>6,} '
      f'{features["Menopause_Status"].isna().sum():>6,}  '
      f'{s} surgical menopause ({s/len(features)*100:.1f}%)')

# Age_Menopause
v = features['Age_Menopause'].notna().sum()
print(f'  {"Age_Menopause":<22} {v:>6,} '
      f'{len(features)-v:>6,}  '
      f'Mean {features["Age_Menopause"].mean():.1f} yrs  '
      f'(surgical only)')

# Reproductive_Span
v = features['Reproductive_Span'].notna().sum()
print(f'  {"Reproductive_Span":<22} {v:>6,} '
      f'{len(features)-v:>6,}  '
      f'Mean {features["Reproductive_Span"].mean():.1f} yrs  '
      f'(surgical only)')

# Validation
assert features['Age_Menopause'].notna().sum() <= \
    (features['Menopause_Status'] == 'Surgical_Menopause').sum(), \
    'Age_Menopause assigned to non-surgical women'
assert (features.loc[
    features['Menopause_Status'] == 'Premenopausal', 
    'Age_Menopause']).isna().all(), \
    'Premenopausal women have Age_Menopause values'

print(f'\n✓ All Domain 1 assertions passed')

Age_Menarche construction:
  From RHQ010 (exact age):         1,519
  From RHQ020 (ordinal imputed):   6
  Pre-menarche (RHQ010=0) → NaN:  1
  Total valid:                     1,525
  Total NaN:                       78

Irregular_Periods construction:
  Raw No to RHQ031:              166
  Excluded — pregnant:           17
  Excluded — breastfeeding:      5
  Excluded — hysterectomy:       53
  Retained — unknown reason:     91
  Final Irregular_Periods = 1:   91

Menopause_Status construction:
  Premenopausal          1,539  (96.0%)
  Surgical_Menopause        63  (3.9%)
  NaN                        1

  Note: Natural_Menopause not derivable — RHD043 code 7
  (menopause) absent in women aged 20–44

════════════════════════════════════════════════════════════
DOMAIN 1 SUMMARY — Menstrual History Features
════════════════════════════════════════════════════════════

  Feature                 Valid    NaN  Key statistic
  ────────────────────── ────── ──────  ───────────────────────────

### Domain 1 results — menstrual history features

All five menstrual history features were derived successfully. Several 
engineering decisions required careful use of the codebook — particularly 
for Irregular_Periods and Menopause_Status, where raw variable values alone 
were insufficient.

**Age_Menarche** — 1,525 of 1,603 women (95.1%) have a valid menarche age. 
The mean of 12.6 years is consistent with published US menarche age 
distributions. Six women were recovered via RHQ020 ordinal midpoint 
imputation — a negligible contribution as anticipated from the coverage 
analysis in notebook 01. One woman with RHQ010 = 0 (primary amenorrhea, 
age 27) has NaN for all menstrual features.

**Irregular_Periods — retained for descriptive purposes only**

This feature captures 91 women (5.7% of the analytical sample) with 
unexplained menstrual irregularity after excluding women with known 
non-CKM explanations (pregnant, breastfeeding, hysterectomy). The 
remaining cases likely represent a heterogeneous mix of causes — 
hormonal contraception, PCOS, hypothalamic amenorrhea, thyroid dysfunction, 
hyperprolactinemia, and other medications affecting the hypothalamic-
pituitary-ovarian axis — that cannot be fully distinguished from available 
data. Complete characterization of menstrual irregularity cause would require clinical 
data beyond what any population survey can capture.

At 5.7% prevalence with an unresolvable heterogeneous confound, this 
feature has insufficient signal for clustering. It is retained in the 
feature matrix for descriptive subgroup analysis in notebook 04 but 
explicitly excluded from the clustering feature set in notebook 05.

**Menopause_Status** — restricted to two categories after discovering that 
RHD043 code 7 (menopause/change of life) is entirely absent in women aged 
20–44. Natural menopause cannot be identified in this analytical sample. 
The 63 women with surgical menopause (3.9%) — defined by hysterectomy 
(RHD280=1) or bilateral oophorectomy (RHQ305=1) — represent a clinically 
distinct group with abrupt estrogen loss, a known risk factor for accelerated 
cardiometabolic dysfunction.

**Age_Menopause and Reproductive_Span** — restricted to the 63 surgically 
menopausal women. Age_Menopause is valid for 62 of 63 (one missing). The 
mean surgical menopause age of 33.4 years reflects the relatively young age 
at which these women underwent surgery. Reproductive_Span is valid for 58 
women — mean 21.2 years, substantially shorter than the population average 
of approximately 37 years — consistent with premature surgical estrogen loss.

**Note for clustering:** Age_Menopause and Reproductive_Span will be sparse 
features in the clustering analysis — calculable for only 58–62 women (3.6–
3.9% of the analytical sample). These features are retained for descriptive 
purposes and subgroup analysis but are unlikely to drive cluster separation 
given their sparsity.

---
## Section 5 · Domain 2: Pregnancy History & Adverse Pregnancy Outcomes

Adverse pregnancy outcomes (APOs) are the central exposure of interest in 
this analysis. The 2023 AHA/ACC CKM framework explicitly recognizes APOs 
as cardiovascular risk enhancers — yet they remain absent from routine 
clinical risk assessment. This domain engineers features that capture both 
pregnancy history and the specific outcomes that carry the strongest 
cardiometabolic signal.

Eight features are derived in this section:

- **Ever_Pregnant** — binary flag, the primary stratification variable
- **Gravidity** — total number of pregnancies (RHQ160, top-coded at 11)
- **Parity** — total number of deliveries (RHD167, top-coded at 5)
- **Pregnancy_Loss** — difference between gravidity and parity, capturing 
  the burden of pregnancy loss (miscarriage, stillbirth, ectopic pregnancy). 
  High gravidity combined with low parity signals repeated pregnancy failure, 
  independently associated with thrombophilia, inflammation, and adverse 
  cardiometabolic risk.
- **Nulliparous** — binary flag for women who were pregnant but never 
  delivered — a clinically distinct group
- **Has_GDM** — binary flag for gestational diabetes, the primary APO 
  exposure variable
- **Has_Macrosomia** — binary flag for delivery of a baby weighing 9 lbs 
  or more
- **APO_Score** — composite adverse pregnancy outcome score combining 
  GDM and macrosomia with clinical weighting

### Engineering decisions for this domain

**RHQ160 — Total number of pregnancies (Gravidity):**
Top-coded at 11 (code 11 = 11 or more pregnancies). Treated as 11 for 
continuous use — a conservative approach that slightly underestimates 
gravidity for women with 12+ pregnancies. In the analytical sample of 
women aged 20–44 this affects very few participants.

**RHD167 — Total number of deliveries (Parity):**
Top-coded at 5 (code 5 = 5 or more deliveries). Treated as 5 for 
continuous use. Zero values are clinically meaningful — women who were 
pregnant but never delivered (nulliparous by outcome). These were 
correctly preserved by the SAS artifact fix in notebook 01.

**Pregnancy_Loss — Gravidity minus Parity:**
Derived from Gravidity and Parity rather than directly from a single 
NHANES variable. Gravidity and Parity are highly correlated (r=0.78, 
confirmed in notebook 04 exploratory analysis) — including both in 
clustering would double-weight pregnancy burden in distance calculations. 
Pregnancy_Loss captures the unique information Gravidity contributes 
beyond Parity: the gap between pregnancies attempted and pregnancies 
delivered. A value of zero indicates no pregnancy loss. Higher values 
indicate greater loss burden. This feature is retained for descriptive 
and exposure analysis rather than as a clustering input — in the current 
pipeline design, clustering (notebooks 05–06) runs on CKM biomarkers 
only, and reproductive history is tested for cluster enrichment 
afterward in notebook 07.

**Nulliparous definition:**
A woman is classified as nulliparous if she was ever pregnant (RHQ131=1) 
but has zero deliveries (RHD167=0). This is distinct from never having 
been pregnant. Nulliparous women who experienced pregnancy loss may have 
different cardiometabolic risk profiles than women who were never pregnant.

**Has_GDM — RHQ162:**
Already recoded in Section 3 including the conservative reclassification 
of borderline GDM (code 3 → Yes). Only asked of women who confirmed 
prior pregnancy via RHQ131. NaN for never-pregnant women is structural.

**Has_Macrosomia — RHQ172:**
Only asked of women who confirmed prior pregnancy. NaN for never-pregnant 
women is structural.

**APO_Score — composite:**
Weighted composite score reflecting cumulative adverse pregnancy outcome 
burden:

| Component | Weight | Clinical rationale |
|---|---|---|
| Has_GDM | 2 | Strongest cardiometabolic signal — insulin resistance proxy |
| Has_Macrosomia | 1 | Independent APO marker — fetal overgrowth signal |

Score range: 0 (no APOs) to 3 (GDM + macrosomia). Women who were never 
pregnant receive NaN — the score is only meaningful for ever-pregnant women. 
Higher scores indicate greater cumulative APO burden and hypothetically 
greater cardiometabolic risk. These weights reflect relative severity as a modeling
simplification, not a validated clinical scoring system.

Note: Hypertensive disorders of pregnancy (preeclampsia, gestational 
hypertension) — which carry the strongest cardiovascular risk signal — 
are absent from the P_RHQ 2017–March 2020 public use file. The APO_Score 
therefore captures metabolic APOs only. This is a structural limitation 
of the dataset documented in notebook 01.

In [6]:
# ── Domain 2: Pregnancy history & adverse pregnancy outcomes ───────────────
# Features derived in this section:
#   Ever_Pregnant   — binary, primary stratification variable
#   Gravidity       — total pregnancies (top-coded at 11)
#   Parity          — total deliveries (top-coded at 5)
#   Nulliparous     — pregnant but never delivered
#   Has_GDM         — gestational diabetes (primary APO exposure)
#   Has_Macrosomia  — baby ≥9 lbs
#   APO_Score       — weighted composite APO burden score

# ── Feature 1: Ever_Pregnant ───────────────────────────────────────────────
# Already recoded in Section 3 (1.0=Yes, 0.0=No, NaN=unknown)
# Two women have NaN: 1 refused (age 37), 1 pre-menarche (age 27)
# Used directly from df without transformation

features['Ever_Pregnant'] = df['RHQ131']

print('Ever_Pregnant:')
print(f'  Yes (1.0): {(features["Ever_Pregnant"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Ever_Pregnant"] == 0.0).sum():,}')
print(f'  NaN:       {features["Ever_Pregnant"].isna().sum():,}')

# ── Feature 2: Gravidity ───────────────────────────────────────────────────
# RHQ160 — total number of pregnancies
# Top-coded: code 11 = 11 or more pregnancies → treated as 11
# Only asked of ever-pregnant women → NaN for never-pregnant is structural
# Missing codes (77, 99) already cleaned to NaN in notebook 01

gravidity = df['RHQ160'].copy().astype(float)

# Validate top-coding
n_topcoded_grav = (gravidity == 11.0).sum()

features['Gravidity'] = gravidity

print(f'\nGravidity:')
print(f'  Valid:       {features["Gravidity"].notna().sum():,}')
print(f'  NaN:         {features["Gravidity"].isna().sum():,}')
print(f'  Top-coded (11+): {n_topcoded_grav:,}')
print(f'  Mean:        {features["Gravidity"].mean():.1f}')
print(f'  Range:       {features["Gravidity"].min():.0f}–'
      f'{features["Gravidity"].max():.0f}')

# ── Feature 3: Parity ──────────────────────────────────────────────────────
# RHD167 — total number of deliveries
# Top-coded: code 5 = 5 or more deliveries → treated as 5
# Code 0 = nulliparous (pregnant but no deliveries) — clinically meaningful
# Preserved by SAS artifact fix in notebook 01

parity = df['RHD167'].copy().astype(float)

# Validate top-coding and zero values
n_topcoded_par = (parity == 5.0).sum()
n_zero_parity  = (parity == 0.0).sum()

features['Parity'] = parity

print(f'\nParity:')
print(f'  Valid:           {features["Parity"].notna().sum():,}')
print(f'  NaN:             {features["Parity"].isna().sum():,}')
print(f'  Zero (nulliparous): {n_zero_parity:,}')
print(f'  Top-coded (5+):  {n_topcoded_par:,}')
print(f'  Mean:            {features["Parity"].mean():.1f}')
print(f'  Range:           {features["Parity"].min():.0f}–'
      f'{features["Parity"].max():.0f}')

# ── Feature 4: Nulliparous ─────────────────────────────────────────────────
# Ever pregnant (RHQ131=1) AND zero deliveries (RHD167=0)
# Clinically distinct from never-pregnant
# Represents women who experienced pregnancy loss only

ever_pregnant_mask = features['Ever_Pregnant'] == 1.0
nulliparous_mask   = ever_pregnant_mask & (features['Parity'] == 0.0)

features['Nulliparous'] = np.where(
    features['Ever_Pregnant'].isna(), np.nan,
    np.where(nulliparous_mask, 1.0, 0.0)
)

print(f'\nNulliparous (pregnant but no deliveries):')
print(f'  Nulliparous (1.0): {(features["Nulliparous"] == 1.0).sum():,}')
print(f'  Parous (0.0):      {(features["Nulliparous"] == 0.0).sum():,}')
print(f'  NaN:               {features["Nulliparous"].isna().sum():,}')

# ── Feature 5: Has_GDM ─────────────────────────────────────────────────────
# RHQ162 — already recoded in Section 3
# Includes borderline GDM reclassified as Yes (6 women)
# Primary APO exposure variable for CKM clustering
# NaN for never-pregnant women is structural

features['Has_GDM'] = df['RHQ162']

print(f'\nHas_GDM:')
print(f'  Yes (1.0): {(features["Has_GDM"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Has_GDM"] == 0.0).sum():,}')
print(f'  NaN:       {features["Has_GDM"].isna().sum():,}')
print(f'  Prevalence among ever-pregnant: '
      f'{(features["Has_GDM"] == 1.0).sum() / ever_pregnant_mask.sum() * 100:.1f}%')

# ── Feature 6: Has_Macrosomia ──────────────────────────────────────────────
# RHQ172 — baby weighing 9 lbs or more at birth
# Already recoded in Section 3
# NaN for never-pregnant women is structural

features['Has_Macrosomia'] = df['RHQ172']

print(f'\nHas_Macrosomia:')
print(f'  Yes (1.0): {(features["Has_Macrosomia"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Has_Macrosomia"] == 0.0).sum():,}')
print(f'  NaN:       {features["Has_Macrosomia"].isna().sum():,}')
print(f'  Prevalence among ever-pregnant: '
      f'{(features["Has_Macrosomia"] == 1.0).sum() / ever_pregnant_mask.sum() * 100:.1f}%')

# ── Feature 7: APO_Score ───────────────────────────────────────────────────
# Weighted composite of metabolic APOs
# GDM weight = 2 (strongest cardiometabolic signal)
# Macrosomia weight = 1 (independent APO marker)
# Score range: 0 (no APOs) to 3 (GDM + macrosomia)
# NaN for never-pregnant women — score meaningless without pregnancy history
#
# Note: Hypertensive disorders of pregnancy (preeclampsia, gestational
# hypertension) absent from P_RHQ 2017-March 2020 public use file
# APO_Score captures metabolic APOs only

# Only calculate for ever-pregnant women with both APO variables known
gdm_known  = features['Has_GDM'].notna()
macs_known = features['Has_Macrosomia'].notna()
apo_mask   = ever_pregnant_mask & gdm_known & macs_known

apo_score = np.where(
    ~apo_mask, np.nan,
    (features['Has_GDM'].fillna(0) * 2) +
    (features['Has_Macrosomia'].fillna(0) * 1)
)

features['APO_Score'] = apo_score

print(f'\nAPO_Score:')
print(f'  Valid (ever-pregnant with both APOs known): '
      f'{features["APO_Score"].notna().sum():,}')
print(f'  NaN: {features["APO_Score"].isna().sum():,}')
print(f'\n  Score distribution:')
for score in [0.0, 1.0, 2.0, 3.0]:
    n   = (features['APO_Score'] == score).sum()
    pct = n / features['APO_Score'].notna().sum() * 100
    label = {
        0.0: 'No APOs',
        1.0: 'Macrosomia only',
        2.0: 'GDM only',
        3.0: 'GDM + Macrosomia'
    }[score]
    print(f'  {score:.0f} — {label:<20} {n:>5,}  ({pct:.1f}%)')

# ── Feature 8: Pregnancy_Loss ──────────────────────────────────────────────
# Gravidity minus Parity — captures pregnancy loss burden
# High Gravidity + Low Parity = repeated pregnancy loss
# Independently associated with thrombophilia, inflammation, and CVD risk
# Only calculated for ever-pregnant women with both values known
# Zero = no losses, higher values = greater loss burden

ever_preg_mask = features['Ever_Pregnant'] == 1.0
both_known     = features['Gravidity'].notna() & features['Parity'].notna()

features['Pregnancy_Loss'] = np.nan
features.loc[ever_preg_mask & both_known, 'Pregnancy_Loss'] = (
    features.loc[ever_preg_mask & both_known, 'Gravidity'] -
    features.loc[ever_preg_mask & both_known, 'Parity']
)

# Validate — no negative values
assert (features['Pregnancy_Loss'].dropna() >= 0).all(), \
    'Negative Pregnancy_Loss values — check Gravidity and Parity'

print(f'\nPregnancy_Loss:')
print(f'  Valid:    {features["Pregnancy_Loss"].notna().sum():,}')
print(f'  NaN:      {features["Pregnancy_Loss"].isna().sum():,}')
print(f'  Mean:     {features["Pregnancy_Loss"].mean():.2f}')
print(f'  Max:      {features["Pregnancy_Loss"].max():.0f}')
print(f'  Loss > 0: {(features["Pregnancy_Loss"] > 0).sum():,} '
      f'({(features["Pregnancy_Loss"] > 0).sum() / features["Pregnancy_Loss"].notna().sum() * 100:.1f}% of valid)')
print(f'✓ No negative values confirmed')

# ── Domain 2 summary ───────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print(f'DOMAIN 2 SUMMARY — Pregnancy History & APO Features')
print(f'{"═"*60}\n')

print(f'  {"Feature":<18} {"Valid":>6} {"NaN":>6}  {"Key statistic"}')
print(f'  {"─"*18} {"─"*6} {"─"*6}  {"─"*35}')

for feat, stat in [
    ('Ever_Pregnant',  f'{(features["Ever_Pregnant"]==1.0).sum():,} Yes '
                       f'({(features["Ever_Pregnant"]==1.0).sum()/len(features)*100:.1f}%)'),
    ('Gravidity',      f'Mean {features["Gravidity"].mean():.1f}  '
                       f'Range {features["Gravidity"].min():.0f}–'
                       f'{features["Gravidity"].max():.0f}'),
    ('Parity',         f'Mean {features["Parity"].mean():.1f}  '
                       f'Range {features["Parity"].min():.0f}–'
                       f'{features["Parity"].max():.0f}'),
    ('Nulliparous',    f'{(features["Nulliparous"]==1.0).sum():,} women '
                       f'({(features["Nulliparous"]==1.0).sum()/ever_pregnant_mask.sum()*100:.1f}% of pregnant)'),
    ('Has_GDM',        f'{(features["Has_GDM"]==1.0).sum():,} Yes '
                       f'({(features["Has_GDM"]==1.0).sum()/ever_pregnant_mask.sum()*100:.1f}% of pregnant)'),
    ('Has_Macrosomia', f'{(features["Has_Macrosomia"]==1.0).sum():,} Yes '
                       f'({(features["Has_Macrosomia"]==1.0).sum()/ever_pregnant_mask.sum()*100:.1f}% of pregnant)'),
    ('APO_Score',      f'Mean {features["APO_Score"].mean():.2f}  '
                       f'Range {features["APO_Score"].min():.0f}–'
                       f'{features["APO_Score"].max():.0f}'),
    ('Pregnancy_Loss', f'Mean {features["Pregnancy_Loss"].mean():.2f}  '
                       f'Max {features["Pregnancy_Loss"].max():.0f}  '
                       f'({(features["Pregnancy_Loss"] > 0).sum():,} with loss)'),
]:
    v = features[feat].notna().sum()
    n = features[feat].isna().sum()
    print(f'  {feat:<18} {v:>6,} {n:>6,}  {stat}')

# ── Validation ─────────────────────────────────────────────────────────────
# APO features should only have values for ever-pregnant women
assert features.loc[
    features['Ever_Pregnant'] == 0.0, 'Has_GDM'
].isna().all(), 'Has_GDM assigned to never-pregnant women'

assert features.loc[
    features['Ever_Pregnant'] == 0.0, 'APO_Score'
].isna().all(), 'APO_Score assigned to never-pregnant women'

assert features.loc[
    features['Nulliparous'] == 1.0, 'Parity'
].eq(0.0).all(), 'Nulliparous women have non-zero parity'

print(f'\n✓ All Domain 2 assertions passed')

Ever_Pregnant:
  Yes (1.0): 1,150
  No  (0.0): 451
  NaN:       2

Gravidity:
  Valid:       1,114
  NaN:         489
  Top-coded (11+): 4
  Mean:        3.0
  Range:       1–11

Parity:
  Valid:           1,135
  NaN:             468
  Zero (nulliparous): 72
  Top-coded (5+):  66
  Mean:            2.2
  Range:           0–5

Nulliparous (pregnant but no deliveries):
  Nulliparous (1.0): 72
  Parous (0.0):      1,529
  NaN:               2

Has_GDM:
  Yes (1.0): 135
  No  (0.0): 1,013
  NaN:       455
  Prevalence among ever-pregnant: 11.7%

Has_Macrosomia:
  Yes (1.0): 144
  No  (0.0): 917
  NaN:       542
  Prevalence among ever-pregnant: 12.5%

APO_Score:
  Valid (ever-pregnant with both APOs known): 1,061
  NaN: 542

  Score distribution:
  0 — No APOs                813  (76.6%)
  1 — Macrosomia only        114  (10.7%)
  2 — GDM only               104  (9.8%)
  3 — GDM + Macrosomia        30  (2.8%)

Pregnancy_Loss:
  Valid:    1,100
  NaN:      503
  Mean:     0.82
  Max:      

In [7]:
# Investigate Gravidity NaN among ever-pregnant women
ever_preg = features['Ever_Pregnant'] == 1.0
grav_missing = ever_preg & features['Gravidity'].isna()

print(f'Ever-pregnant women missing Gravidity: {grav_missing.sum():,}')
print(f'\nRHQ160 raw values for these women:')
print(df.loc[grav_missing, 'RHQ160'].value_counts(dropna=False))

print(f'\nCurrently pregnant (RHD143) among missing Gravidity:')
print(df.loc[grav_missing, 'RHD143'].value_counts(dropna=False))

Ever-pregnant women missing Gravidity: 36

RHQ160 raw values for these women:
RHQ160
NaN    36
Name: count, dtype: int64

Currently pregnant (RHD143) among missing Gravidity:
RHD143
0.0    28
NaN     4
1.0     4
Name: count, dtype: int64


### Domain 2 results — pregnancy history and APO features

All eight features derived successfully with clinically plausible 
distributions. All three assertions passed — APO features contain no 
values for never-pregnant women and nulliparous women correctly show 
zero parity.

**Ever_Pregnant** — 1,150 women (71.7%) have been pregnant, consistent 
with the weighted prevalence estimate from notebook 01 (71.8%) confirming 
feature construction is correct. Two women have unknown pregnancy status — 
one who refused RHQ131 (age 37) and one with primary amenorrhea (age 27) 
routed out before RHQ131.

**Gravidity and Parity** — among ever-pregnant women, mean gravidity is 
3.0 pregnancies and mean parity is 2.2 deliveries, consistent with US 
fertility patterns for women aged 20–44. Four women with top-coded 
gravidity (11+) and 66 with top-coded parity (5+) are retained at their 
top-coded values — a conservative approach that slightly underestimates 
counts for the highest-parity women. Thirty-six ever-pregnant women are 
missing gravidity counts (item non-response to RHQ160) — not recoverable 
but does not affect APO features.

**Pregnancy_Loss** — 1,100 women have both Gravidity and Parity recorded, 
allowing calculation of pregnancy loss burden **(Gravidity − Parity)**. Of 
these, 542 women (49.3%) experienced at least one pregnancy loss, with a 
mean of 0.82 and a maximum of 10. This feature captures the clinically 
important signal of repeated pregnancy failure — high gravidity combined 
with low parity — which is independently associated with thrombophilia, 
chronic inflammation, and adverse cardiometabolic risk. Gravidity and 
Parity are highly correlated (r=0.78, identified in notebook 04 
exploratory analysis) — Pregnancy_Loss is retained over Gravidity for 
descriptive analysis, preserving the unique information Gravidity 
contributes beyond Parity while avoiding collinearity. Neither feature 
is used as a clustering input in the current design.

**Nulliparous** — 72 women (6.3% of ever-pregnant) were pregnant but 
never delivered. This clinically distinct group likely experienced 
pregnancy loss — miscarriage, ectopic pregnancy, or termination. Their 
cardiometabolic profile may differ from both never-pregnant and parous 
women and will be examined descriptively in notebook 04.

**Has_GDM** — 135 women (11.7% of ever-pregnant) have a history of 
gestational diabetes, including 6 women with borderline GDM conservatively
reclassified as Yes. This prevalence is consistent with published NHANES 
estimates (10–15%) providing independent validation of the feature construction.
One woman has known GDM status but unknown macrosomia status, and is excluded 
from the APO_Score-based count below.

**Has_Macrosomia** — 144 women (12.5% of ever-pregnant) delivered at 
least one baby weighing 9 lbs or more. This is slightly higher than the 
general population estimate of 8–10% — consistent with the demographic 
composition of the analytical sample which overrepresents groups with 
higher rates of macrosomia.

**APO_Score — the primary exposure variable:**

| Score | Label | N | % of scored |
|---|---|---|---|
| 0 | No APOs | 813 | 76.6% |
| 1 | Macrosomia only | 114 | 10.7% |
| 2 | GDM only | 104 | 9.8% |
| 3 | GDM + Macrosomia | 30 | 2.8% |
| — | NaN (never pregnant or missing) | 542 | — |

The 134 women with GDM history (APO_Score ≥ 2, representing 12.6% of ever-pregnant
women) are the primary exposure group examined against the CKM biomarker-only
clusters formed in notebooks 05–06 — one fewer than the 135 reported above for
Has_GDM alone, since APO_Score requires macrosomia status to also be known and
one woman with confirmed GDM has that value missing. That enrichment test is
performed in notebook 07, after the clusters have already been formed without
any reference to APO history — the central question of this project.

**Important limitation:** Hypertensive disorders of pregnancy — 
preeclampsia and gestational hypertension — which carry the strongest 
long-term cardiovascular risk signal, are absent from the P_RHQ 
2017–March 2020 public use file. The APO_Score therefore captures 
metabolic APOs only. This structural limitation is documented in 
notebook 01 and constrains the scope of the APO-CKM analysis to 
gestational diabetes and macrosomia as the primary exposure variables.

---
## Section 6 · Domain 3: Surgical History Features

Surgical history captures permanent alterations to the reproductive system 
that directly affect hormonal status and cardiometabolic risk. Hysterectomy 
and bilateral oophorectomy are the two most clinically relevant surgical 
procedures in this context — oophorectomy in particular causes abrupt 
surgical menopause with immediate estrogen loss, which is associated with 
accelerated cardiometabolic dysfunction compared to gradual natural menopause.

Three features are derived in this section:

- **Has_Hysterectomy** — binary flag for uterine removal (RHD280)
- **Has_Oophorectomy** — binary flag for bilateral ovary removal (RHQ305)
- **Surgical_Menopause** — binary composite flag combining hysterectomy 
  and oophorectomy, equivalent to Menopause_Status == Surgical_Menopause 
  from Domain 1 but expressed as a binary for clustering use

### Engineering decisions for this domain

**Has_Hysterectomy and Has_Oophorectomy:**
Both already recoded in Section 3 (1.0=Yes, 0.0=No). Used directly 
without transformation. NaN values reflect women who were not asked 
these questions — asked only of women aged 20–150 in the full sample, 
but structural missingness in the analytical sample is minimal.

**Surgical_Menopause:**
Defined as hysterectomy (RHD280=1) OR bilateral oophorectomy (RHQ305=1). 
This is consistent with the Menopause_Status derivation in Domain 1 an

In [8]:
# ── Domain 3: Surgical history features ───────────────────────────────────
# Features derived in this section:
#   Has_Hysterectomy  — binary flag for uterine removal
#   Has_Oophorectomy  — binary flag for bilateral ovary removal
#   Surgical_Menopause — binary composite (hysterectomy OR oophorectomy)
#   Age_Oophorectomy  — age at oophorectomy (descriptive only, sparse)

# ── Feature 1: Has_Hysterectomy ───────────────────────────────────────────
# RHD280 — already recoded in Section 3 (1.0=Yes, 0.0=No)
# Asked of women aged 20–150 — NaN minimal in analytical sample

features['Has_Hysterectomy'] = df['RHD280']

print('Has_Hysterectomy:')
print(f'  Yes (1.0): {(features["Has_Hysterectomy"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Has_Hysterectomy"] == 0.0).sum():,}')
print(f'  NaN:       {features["Has_Hysterectomy"].isna().sum():,}')
print(f'  Prevalence: '
      f'{(features["Has_Hysterectomy"] == 1.0).sum() / len(features) * 100:.1f}%')

# ── Feature 2: Has_Oophorectomy ───────────────────────────────────────────
# RHQ305 — already recoded in Section 3 (1.0=Yes, 0.0=No)
# Asked of all women — bilateral oophorectomy only

features['Has_Oophorectomy'] = df['RHQ305']

print(f'\nHas_Oophorectomy:')
print(f'  Yes (1.0): {(features["Has_Oophorectomy"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Has_Oophorectomy"] == 0.0).sum():,}')
print(f'  NaN:       {features["Has_Oophorectomy"].isna().sum():,}')
print(f'  Prevalence: '
      f'{(features["Has_Oophorectomy"] == 1.0).sum() / len(features) * 100:.1f}%')

# ── Feature 3: Surgical_Menopause ─────────────────────────────────────────
# Binary composite: hysterectomy (RHD280=1) OR oophorectomy (RHQ305=1)
# Consistent with Menopause_Status derivation in Domain 1
# Binary version for clustering use

hyst_mask  = features['Has_Hysterectomy'] == 1.0
ooph_mask  = features['Has_Oophorectomy'] == 1.0
surg_mask  = hyst_mask | ooph_mask

# NaN only if both source variables are NaN
both_nan   = features['Has_Hysterectomy'].isna() & \
             features['Has_Oophorectomy'].isna()

features['Surgical_Menopause'] = np.where(
    both_nan, np.nan,
    np.where(surg_mask, 1.0, 0.0)
)

print(f'\nSurgical_Menopause:')
print(f'  Yes (1.0): {(features["Surgical_Menopause"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Surgical_Menopause"] == 0.0).sum():,}')
print(f'  NaN:       {features["Surgical_Menopause"].isna().sum():,}')
print(f'  Prevalence: '
      f'{(features["Surgical_Menopause"] == 1.0).sum() / len(features) * 100:.1f}%')

# Consistency check with Domain 1 Menopause_Status
n_consistent = (
    (features['Surgical_Menopause'] == 1.0) ==
    (features['Menopause_Status'] == 'Surgical_Menopause')
).sum()
print(f'\n  Consistency with Domain 1 Menopause_Status: '
      f'{n_consistent:,} of {len(features):,} rows agree')

# ── Feature 4: Age_Oophorectomy ───────────────────────────────────────────
# RHQ332 — age when both ovaries removed
# Sparse by design — only asked of women who confirmed oophorectomy
# Retained for descriptive purposes only
# Bottom-coded: values ≤19 stored as 19 in codebook

features['Age_Oophorectomy'] = df['RHQ332']

n_valid_ooph = features['Age_Oophorectomy'].notna().sum()
print(f'\nAge_Oophorectomy (descriptive only):')
print(f'  Valid:   {n_valid_ooph:,}')
print(f'  NaN:     {features["Age_Oophorectomy"].isna().sum():,}')
if n_valid_ooph > 0:
    print(f'  Mean:    {features["Age_Oophorectomy"].mean():.1f} years')
    print(f'  Range:   {features["Age_Oophorectomy"].min():.0f}–'
          f'{features["Age_Oophorectomy"].max():.0f} years')

# ── Domain 3 summary ───────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print(f'DOMAIN 3 SUMMARY — Surgical History Features')
print(f'{"═"*60}\n')

print(f'  {"Feature":<22} {"Valid":>6} {"NaN":>6}  {"Key statistic"}')
print(f'  {"─"*22} {"─"*6} {"─"*6}  {"─"*30}')

for feat, stat in [
    ('Has_Hysterectomy',
     f'{(features["Has_Hysterectomy"]==1.0).sum():,} Yes '
     f'({(features["Has_Hysterectomy"]==1.0).sum()/len(features)*100:.1f}%)'),
    ('Has_Oophorectomy',
     f'{(features["Has_Oophorectomy"]==1.0).sum():,} Yes '
     f'({(features["Has_Oophorectomy"]==1.0).sum()/len(features)*100:.1f}%)'),
    ('Surgical_Menopause',
     f'{(features["Surgical_Menopause"]==1.0).sum():,} Yes '
     f'({(features["Surgical_Menopause"]==1.0).sum()/len(features)*100:.1f}%)'),
    ('Age_Oophorectomy',
     f'Mean {features["Age_Oophorectomy"].mean():.1f} yrs '
     f'— descriptive only' if n_valid_ooph > 0 else 'sparse — descriptive only'),
]:
    v = features[feat].notna().sum()
    n = features[feat].isna().sum()
    print(f'  {feat:<22} {v:>6,} {n:>6,}  {stat}')

# ── Validation ─────────────────────────────────────────────────────────────
# Surgical_Menopause should be consistent with Domain 1 Menopause_Status
assert (
    features['Surgical_Menopause'] == 1.0
).sum() == (
    features['Menopause_Status'] == 'Surgical_Menopause'
).sum(), 'Surgical_Menopause count inconsistent with Domain 1 Menopause_Status'

# Age_Oophorectomy should only exist for women with oophorectomy
assert features.loc[
    features['Has_Oophorectomy'] == 0.0, 'Age_Oophorectomy'
].isna().all(), 'Age_Oophorectomy assigned to women without oophorectomy'

print(f'\n✓ All Domain 3 assertions passed')

Has_Hysterectomy:
  Yes (1.0): 61
  No  (0.0): 1,523
  NaN:       19
  Prevalence: 3.8%

Has_Oophorectomy:
  Yes (1.0): 15
  No  (0.0): 1,565
  NaN:       23
  Prevalence: 0.9%

Surgical_Menopause:
  Yes (1.0): 63
  No  (0.0): 1,522
  NaN:       18
  Prevalence: 3.9%

  Consistency with Domain 1 Menopause_Status: 1,603 of 1,603 rows agree

Age_Oophorectomy (descriptive only):
  Valid:   15
  NaN:     1,588
  Mean:    35.5 years
  Range:   28–40 years

════════════════════════════════════════════════════════════
DOMAIN 3 SUMMARY — Surgical History Features
════════════════════════════════════════════════════════════

  Feature                 Valid    NaN  Key statistic
  ────────────────────── ────── ──────  ──────────────────────────────
  Has_Hysterectomy        1,584     19  61 Yes (3.8%)
  Has_Oophorectomy        1,580     23  15 Yes (0.9%)
  Surgical_Menopause      1,585     18  63 Yes (3.9%)
  Age_Oophorectomy           15  1,588  Mean 35.5 yrs — descriptive only

✓ All Domain 3 

### Domain 3 results — surgical history features

All four surgical history features derived successfully. The consistency 
assertion confirms perfect agreement between Surgical_Menopause and the 
Menopause_Status derivation in Domain 1 — both approaches identify the 
same 63 women.

**Has_Hysterectomy** — 61 women (3.8%) report a hysterectomy. This 
prevalence is consistent with published rates for women aged 20–44, 
where hysterectomy is substantially less common than in older age groups. 
Hysterectomy at this age typically indicates serious uterine pathology — 
fibroids, adenomyosis, or malignancy — and may be accompanied by 
oophorectomy depending on the indication.

**Has_Oophorectomy** — 15 women (0.9%) report bilateral oophorectomy. 
This is appropriately sparse for a 20–44 age group — bilateral 
oophorectomy at reproductive age is typically performed for conditions 
such as severe endometriosis, hereditary cancer syndrome (BRCA), or 
ovarian torsion. These 15 women experienced abrupt surgical menopause 
with immediate complete estrogen loss — the most severe form of estrogen 
deprivation.

**Surgical_Menopause** — 63 women (3.9%) classified as surgically 
menopausal via hysterectomy or oophorectomy or both. This binary feature 
is the clustering-ready version of Menopause_Status from Domain 1, 
confirmed consistent across both derivations.

**Age_Oophorectomy** — valid for all 15 oophorectomy women, mean age 
35.5 years (range 28–40). Too sparse for clustering — retained for 
descriptive subgroup analysis in notebook 04 only.

**Note for clustering:** Has_Hysterectomy, Has_Oophorectomy, and 
Surgical_Menopause are all sparse features (0.9%–3.9%). Surgical_Menopause 
is the preferred feature for clustering — it captures the combined 
surgical estrogen loss signal and has the highest prevalence of the three. 
Has_Hysterectomy and Has_Oophorectomy are retained for descriptive 
purposes and subgroup analysis.

---
## Section 7 · Domain 4: Hormone Therapy Features

Hormone therapy (HT) use in reproductive-age women aged 20–44 is 
clinically distinct from postmenopausal HT. At this age, HT is typically 
prescribed for surgical menopause, premature ovarian insufficiency, or 
specific gynecological conditions — not for menopausal symptom management 
as in older women. Exogenous estrogen use directly modifies cardiometabolic 
risk, making it an important covariate for interpreting CKM biomarkers in 
notebook 03.

Two features are derived in this section:

- **Ever_HRT** — binary flag for ever having used female hormones (RHQ540)
- **HRT_Type** — categorical composite identifying the primary form of 
  hormone therapy used, derived from RHQ542A–D

### Engineering decisions for this domain

**Ever_HRT:**
Used directly from RHQ540, already recoded in Section 3. Only 66 women 
(4.1% of the analytical sample) report ever using hormones — expected for 
reproductive-age women. NaN reflects women not asked this question.

**HRT_Type — RHQ542A–D:**
These four variables use a different coding system from other binary 
variables in P_RHQ — they were not recoded in Section 3 because they use 
codes 10/11 rather than 1/2. RHQ542A–D are multi-select variables where 
each code indicates whether that form was used:

| Variable | Form | Code structure |
|---|---|---|
| RHQ542A | Pills | 10=checked, missing=not checked |
| RHQ542B | Patches | 10=checked, missing=not checked |
| RHQ542C | Cream/suppository/injection | 10=checked, missing=not checked |
| RHQ542D | Other | 10=checked, missing=not checked |

Only asked of the 66 women who answered Yes to RHQ540. A woman can 
select multiple forms. HRT_Type is built as a categorical variable 
prioritizing the most clinically relevant form — pills are the most 
common and have the most established cardiometabolic risk data.

**Sparsity note:** With only 66 hormone users in the analytical sample 
(4.1%), both hormone therapy features fall below the 5% prevalence 
threshold established for clustering features. At this prevalence, 
binary features have insufficient variance to meaningfully influence 
distance-based cluster assignment in k-means clustering. Both Ever_HRT 
and HRT_Type are therefore retained for descriptive purposes only.

The more important role of hormone therapy in this analysis is as a 
**confounder** for CKM biomarker interpretation in notebook 03 — 
exogenous estrogen modifies lipid profiles, blood pressure, and glucose 
metabolism. Women on HRT will be flagged in notebook 03 before clustering 
to avoid misattributing biomarker differences to reproductive history 
rather than current hormone exposure.

In [9]:
# ── Domain 4: Hormone therapy features ────────────────────────────────────
# Features derived in this section:
#   Ever_HRT  — binary flag for ever using female hormones
#   HRT_Type  — categorical composite of hormone therapy form

# ── Feature 1: Ever_HRT ───────────────────────────────────────────────────
# RHQ540 — already recoded in Section 3 (1.0=Yes, 0.0=No)
# 66 women (4.1%) report ever using hormones

features['Ever_HRT'] = df['RHQ540']

print('Ever_HRT:')
print(f'  Yes (1.0): {(features["Ever_HRT"] == 1.0).sum():,}')
print(f'  No  (0.0): {(features["Ever_HRT"] == 0.0).sum():,}')
print(f'  NaN:       {features["Ever_HRT"].isna().sum():,}')
print(f'  Prevalence: '
      f'{(features["Ever_HRT"] == 1.0).sum() / len(features) * 100:.1f}%')

# ── Feature 2: HRT_Type ───────────────────────────────────────────────────
# Multi-select variables RHQ542A–D use code 10=checked, NaN=not checked
# Only meaningful for the 66 women who answered Yes to RHQ540
# Priority hierarchy for categorical assignment:
#   1. Pills (RHQ542A) — most common, most established risk data
#   2. Patches (RHQ542B)
#   3. Cream/suppository/injection (RHQ542C)
#   4. Other (RHQ542D)
#   5. Unknown — used HRT but no form specified
#
# Women may use multiple forms — priority captures dominant form

hrt_users = features['Ever_HRT'] == 1.0

# Check raw values for RHQ542A–D among HRT users
print(f'\nRHQ542A–D raw values among {hrt_users.sum()} HRT users:')
for var in ['RHQ542A', 'RHQ542B', 'RHQ542C', 'RHQ542D']:
    n_checked = (df.loc[hrt_users, var] == 10.0).sum()
    print(f'  {var}: {n_checked:,} checked (code 10)')

# Build HRT_Type
hrt_type = pd.Series(np.nan, index=df.index, dtype=object)

# Only assign to HRT users
hrt_type[hrt_users] = 'Unknown'

# Assign in reverse priority so highest priority overwrites
hrt_type[hrt_users & (df['RHQ542D'] == 10.0)] = 'Other'
hrt_type[hrt_users & (df['RHQ542C'] == 10.0)] = 'Cream_Suppository_Injection'
hrt_type[hrt_users & (df['RHQ542B'] == 10.0)] = 'Patches'
hrt_type[hrt_users & (df['RHQ542A'] == 10.0)] = 'Pills'

features['HRT_Type'] = hrt_type

print(f'\nHRT_Type distribution (among {hrt_users.sum()} HRT users):')
for category in ['Pills', 'Patches', 'Cream_Suppository_Injection',
                 'Other', 'Unknown']:
    n = (features['HRT_Type'] == category).sum()
    if n > 0:
        pct = n / hrt_users.sum() * 100
        print(f'  {category:<30} {n:>4,}  ({pct:.1f}%)')
print(f'  {"NaN (non-users)":<30} '
      f'{features["HRT_Type"].isna().sum():>4,}')

# ── Domain 4 summary ───────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print(f'DOMAIN 4 SUMMARY — Hormone Therapy Features')
print(f'{"═"*60}\n')

print(f'  {"Feature":<15} {"Valid":>6} {"NaN":>6}  {"Key statistic"}')
print(f'  {"─"*15} {"─"*6} {"─"*6}  {"─"*35}')

for feat, stat in [
    ('Ever_HRT',
     f'{(features["Ever_HRT"]==1.0).sum():,} Yes '
     f'({(features["Ever_HRT"]==1.0).sum()/len(features)*100:.1f}%)'),
    ('HRT_Type',
     f'{hrt_users.sum():,} users — '
     f'pills most common form'),
]:
    v = features[feat].notna().sum()
    n = features[feat].isna().sum()
    print(f'  {feat:<15} {v:>6,} {n:>6,}  {stat}')

# ── Validation ─────────────────────────────────────────────────────────────
# HRT_Type should only be non-null for Ever_HRT users
assert features.loc[
    features['Ever_HRT'] == 0.0, 'HRT_Type'
].isna().all(), 'HRT_Type assigned to non-users'

# All HRT users should have an HRT_Type
assert features.loc[
    hrt_users, 'HRT_Type'
].notna().all(), 'Some HRT users missing HRT_Type'

print(f'\n✓ All Domain 4 assertions passed')

Ever_HRT:
  Yes (1.0): 66
  No  (0.0): 1,531
  NaN:       6
  Prevalence: 4.1%

RHQ542A–D raw values among 66 HRT users:
  RHQ542A: 43 checked (code 10)
  RHQ542B: 0 checked (code 10)
  RHQ542C: 0 checked (code 10)
  RHQ542D: 0 checked (code 10)

HRT_Type distribution (among 66 HRT users):
  Pills                            43  (65.2%)
  Unknown                          23  (34.8%)
  NaN (non-users)                1,537

════════════════════════════════════════════════════════════
DOMAIN 4 SUMMARY — Hormone Therapy Features
════════════════════════════════════════════════════════════

  Feature          Valid    NaN  Key statistic
  ─────────────── ────── ──────  ───────────────────────────────────
  Ever_HRT         1,597      6  66 Yes (4.1%)
  HRT_Type            66  1,537  66 users — pills most common form

✓ All Domain 4 assertions passed


### Domain 4 results — hormone therapy features

Both hormone therapy features derived successfully with all assertions 
passed.

**Ever_HRT** — 66 women (4.1%) report ever using female hormones. This 
low prevalence is expected for reproductive-age women aged 20–44, where 
hormone therapy is typically prescribed for surgical menopause, premature 
ovarian insufficiency, or specific gynecological conditions rather than 
for menopausal symptom management. Six NaN values reflect women not asked 
this question.

**HRT_Type** — among the 66 hormone users, 43 (65.2%) used pills as their 
primary form. Patches, cream/suppository/injection, and other forms show 
zero users in this analytical sample — an unexpected finding that may 
reflect either genuine pill predominance in this age group or incomplete 
capture of non-pill form selection in this release. Twenty-three women 
(34.8%) are classified as Unknown — they confirmed hormone use but no 
specific form was recorded.

**Note for clustering:**

At 4.1% prevalence, Ever_HRT is marginal for clustering — it is retained 
in the feature matrix but its contribution to cluster separation will be 
limited. HRT_Type is retained for descriptive subgroup analysis in 
notebook 04 only. The clinical importance of hormone therapy is primarily 
as a **confounder** for CKM biomarker interpretation in notebook 03 — 
exogenous estrogen affects lipid profiles, blood pressure, and glucose 
metabolism, and women on HRT should be flagged before clustering to avoid 
misattributing biomarker differences to reproductive history rather than 
current hormone exposure.

---
## Section 8 · Feature Matrix Assembly

All four reproductive health domains are now complete. This section 
assembles the full feature matrix, documents missingness by feature, 
and formally classifies each feature by its intended use in the 
downstream pipeline.

Three categories of intended use are defined:

- **Clustering** — sufficient prevalence, clean signal, suitable for 
  inclusion in the unsupervised clustering in notebook 05
- **Descriptive** — retained for subgroup analysis and visualization in 
  notebook 04 but excluded from clustering due to sparsity, confounding, 
  or limited signal
- **Stratification** — used to define subgroups for analysis rather than 
  as clustering features (e.g. Ever_Pregnant defines who is included in 
  APO analyses)

### Clustering design

The CKM clustering analysis (notebooks 05–06) runs on biomarkers only — 
HbA1c, BMI, mean systolic blood pressure, eGFR, HDL cholesterol, and 
glucose — deliberately excluding reproductive history from the distance 
calculation.

An earlier version of this pipeline classified four reproductive 
features (Age_Menarche, Parity, Pregnancy_Loss, APO_Score) as clustering 
inputs alongside the biomarkers. That approach was abandoned: APO_Score 
is the primary exposure this project tests against cluster membership, 
and using it to help *define* the clusters in the first place would make 
any resulting association circular — the clusters would partly encode 
APO history by construction, and then "discovering" that APO history 
predicts cluster membership would prove nothing.

None of the four reproductive features below are classified as 
clustering inputs in the current design. They are retained as 
descriptive and exposure variables and linked to the resulting 
biomarker-only clusters for enrichment testing in notebook 07 — a 
genuinely independent test, since none of these variables informed the 
clustering itself.

### Feature classification rules

Three rules were applied when these features were evaluated against 
clustering criteria (all four are excluded from clustering under the 
current design, for the reason given above — the notes below describe 
the secondary considerations that would otherwise have governed their 
inclusion):

1. **Prevalence threshold** — binary features below 5% prevalence are 
   excluded. At this prevalence a binary feature has insufficient variance 
   to meaningfully influence distance-based cluster assignment in k-means.

2. **Multicollinearity** — individual APO components (Has_GDM, 
   Has_Macrosomia) are excluded in favor of the composite APO_Score to 
   avoid double-counting the APO signal.

3. **Correlated feature pairs** — where two features measure overlapping 
   constructs (identified in notebook 04 correlation analysis), the more 
   clinically informative feature is retained for descriptive use. 
   Gravidity is replaced by Pregnancy_Loss (Gravidity − Parity), which 
   captures the unique information in Gravidity — pregnancy loss burden — 
   that is not already encoded in Parity. High gravidity combined with 
   low parity signals repeated pregnancy failure, independently 
   associated with thrombophilia, inflammation, and adverse 
   cardiometabolic risk.

**Note on Age_Menarche:** Age_Menarche is a continuous variable with 
95.1% coverage and sufficient variance (range 6–20 years, mean 12.6) that 
it could have supported distance-based clustering on methodological 
grounds. It is not used as a clustering input in the current design, but 
remains a variable of clinical interest — earlier menarche is associated 
with higher adiposity, insulin resistance, and cardiometabolic risk — and 
is examined descriptively in notebook 04.

### Reproductive feature classification

Four reproductive features were evaluated and are retained as 
descriptive/exposure variables rather than clustering inputs:

| Feature | Type | Intended use |
|---|---|---|
| Age_Menarche | Continuous | Descriptive — reproductive timing marker, 95.1% valid |
| Parity | Continuous | Descriptive — delivery burden, 70.8% valid |
| Pregnancy_Loss | Continuous | Descriptive — replaces Gravidity, loss burden |
| APO_Score | Continuous composite | Primary exposure — tested for cluster enrichment in notebook 07, 66.2% valid |

**Note on Gravidity:** Gravidity is highly correlated with Parity 
(r=0.78, identified in notebook 04). Rather than simply dropping it, 
a derived feature Pregnancy_Loss (Gravidity − Parity) preserves the 
unique information Gravidity contributes — the burden of pregnancy loss. 
Gravidity is retained for descriptive analysis only.

These four features are not inputs to the clustering itself. Their 
value lies in what they reveal after clustering: whether biomarker-only 
phenotypes, defined without any reference to pregnancy history, 
nonetheless differ in APO prevalence. That test is performed in 
notebook 07.

This classification is documented here for reference. The actual 
clustering feature set is defined independently in notebook 05, from 
the CKM biomarker candidates established in notebook 03.

In [10]:
# ── CKM biomarker coverage check ──────────────────────────────────────────
# Informs the two-stage clustering design documented above
# Run before finalizing feature classifications

import os

# Load CKM modules
ghb    = pd.read_sas('../data/raw/P_GHB.xpt',
                      format='xport', encoding='utf-8').set_index('SEQN')
bmx    = pd.read_sas('../data/raw/P_BMX.xpt',
                      format='xport', encoding='utf-8').set_index('SEQN')
bpxo   = pd.read_sas('../data/raw/P_BPXO.xpt',
                      format='xport', encoding='utf-8').set_index('SEQN')
biopro = pd.read_sas('../data/raw/P_BIOPRO.xpt',
                      format='xport', encoding='utf-8').set_index('SEQN')
hdl    = pd.read_sas('../data/raw/P_HDL.xpt',
                      format='xport', encoding='utf-8').set_index('SEQN')
trigly = pd.read_sas('../data/raw/P_TRIGLY.xpt',
                      format='xport', encoding='utf-8').set_index('SEQN')

analytical_seqn = features.index

# ── Stage 1 biomarkers — non-fasting ──────────────────────────────────────
stage1_vars = {
    'LBXGH':   (ghb,    'HbA1c (%)'),
    'BMXBMI':  (bmx,    'BMI (kg/m²)'),
    'BMXWAIST':(bmx,    'Waist circumference (cm)'),
    'BPXOSY1': (bpxo,   'Systolic BP (mmHg)'),
    'BPXODI1': (bpxo,   'Diastolic BP (mmHg)'),
    'LBDHDD':  (hdl,    'HDL cholesterol (mg/dL)'),
    'LBXSCR':  (biopro, 'Serum creatinine (mg/dL) → eGFR'),
    'LBXSGL':  (biopro, 'Glucose (mg/dL)'),
}

# ── Stage 2 biomarkers — fasting subsample ─────────────────────────────────
stage2_vars = {
    'LBXTR':   (trigly, 'Triglycerides (mg/dL)'),
    'LBDLDL':  (trigly, 'LDL cholesterol (mg/dL) — calculated'),
}

print('=== CKM Biomarker Coverage in Analytical Sample (n=1,603) ===\n')

print('Stage 1 — Non-fasting biomarkers (full sample):')
print(f'  {"Variable":<12} {"Valid":>6} {"NaN":>6} {"% Valid":>8}  {"Label"}')
print(f'  {"─"*12} {"─"*6} {"─"*6} {"─"*8}  {"─"*35}')

for var, (module, label) in stage1_vars.items():
    if var in module.columns:
        valid = module.loc[
            module.index.isin(analytical_seqn), var
        ].notna().sum()
        nan = len(analytical_seqn) - valid
        pct = valid / len(analytical_seqn) * 100
        flag = ' ⚠️' if pct < 70 else ''
        print(f'  {var:<12} {valid:>6,} {nan:>6,} {pct:>7.1f}%  {label}{flag}')
    else:
        print(f'  {var:<12} {"N/A":>6} {"─":>6} {"─":>8}  '
              f'{label} — variable not found')

print(f'\nStage 2 — Fasting subsample biomarkers:')
print(f'  {"Variable":<12} {"Valid":>6} {"NaN":>6} {"% Valid":>8}  {"Label"}')
print(f'  {"─"*12} {"─"*6} {"─"*6} {"─"*8}  {"─"*35}')

for var, (module, label) in stage2_vars.items():
    if var in module.columns:
        valid = module.loc[
            module.index.isin(analytical_seqn), var
        ].notna().sum()
        nan = len(analytical_seqn) - valid
        pct = valid / len(analytical_seqn) * 100
        flag = ' ⚠️' if pct < 70 else ''
        print(f'  {var:<12} {valid:>6,} {nan:>6,} {pct:>7.1f}%  {label}{flag}')

# Fasting subsample size
fasting_n = (trigly.loc[
    trigly.index.isin(analytical_seqn), 'WTSAFPRP'
] > 0).sum()

print(f'\n  Fasting subsample: {fasting_n:,} of 1,603 women '
      f'({fasting_n/1603*100:.1f}%)')
print(f'  Non-fasting sample: 1,603 women (100.0%)')
print(f'\n  → Two-stage analysis adopted — see markdown above')

=== CKM Biomarker Coverage in Analytical Sample (n=1,603) ===

Stage 1 — Non-fasting biomarkers (full sample):
  Variable      Valid    NaN  % Valid  Label
  ──────────── ────── ────── ────────  ───────────────────────────────────
  LBXGH         1,522     81    94.9%  HbA1c (%)
  BMXBMI        1,596      7    99.6%  BMI (kg/m²)
  BMXWAIST      1,552     51    96.8%  Waist circumference (cm)
  BPXOSY1       1,451    152    90.5%  Systolic BP (mmHg)
  BPXODI1       1,451    152    90.5%  Diastolic BP (mmHg)
  LBDHDD        1,503    100    93.8%  HDL cholesterol (mg/dL)
  LBXSCR        1,499    104    93.5%  Serum creatinine (mg/dL) → eGFR
  LBXSGL        1,497    106    93.4%  Glucose (mg/dL)

Stage 2 — Fasting subsample biomarkers:
  Variable      Valid    NaN  % Valid  Label
  ──────────── ────── ────── ────────  ───────────────────────────────────
  LBXTR           745    858    46.5%  Triglycerides (mg/dL) ⚠️
  LBDLDL          744    859    46.4%  LDL cholesterol (mg/dL) — calculate

All non-fasting biomarkers show coverage above 90% — sufficient for 
the primary clustering analysis without excessive data loss. Fasting 
lipids (triglycerides, LDL) are available for only 46.5% of the sample 
due to the NHANES fasting subsample design, confirming the two-stage 
approach documented above.

In [11]:
# ── Feature matrix assembly ────────────────────────────────────────────────
# Confirm all expected features are present
expected_features = [
    # Domain 1 — Menstrual history
    'Age_Menarche', 'Irregular_Periods', 'Menopause_Status',
    'Age_Menopause', 'Reproductive_Span',
    # Domain 2 — Pregnancy history & APO
    'Ever_Pregnant', 'Gravidity', 'Parity', 'Pregnancy_Loss', 'Nulliparous',
    'Has_GDM', 'Has_Macrosomia', 'APO_Score',
    # Domain 3 — Surgical history
    'Has_Hysterectomy', 'Has_Oophorectomy', 'Surgical_Menopause',
    'Age_Oophorectomy',
    # Domain 4 — Hormone therapy
    'Ever_HRT', 'HRT_Type',
]

missing_features = [f for f in expected_features if f not in features.columns]
if missing_features:
    print(f'⚠️  Missing features: {missing_features}')
else:
    print(f'✓ All {len(expected_features)} expected features present')
print(f'  Feature matrix shape: {features.shape}')

# ── Feature classification ─────────────────────────────────────────────────
# None of the reproductive features below are used as clustering inputs.
# The current pipeline design clusters on CKM biomarkers only (notebook 05) —
# reproductive history, including APO_Score, is held out and tested for
# enrichment in the resulting clusters in notebook 07. This avoids the
# circularity of using APO history to both build and validate the clusters.
#
# Three categories:
#   clustering     → CKM biomarker candidates evaluated in notebooks 03/05 (none here)
#   descriptive    → notebook 04
#   stratification → subgroup definitions

feature_classification = {
    # Domain 1 — Menstrual history
    'Age_Menarche':       'descriptive',    # reproductive timing — exposure variable, not a clustering input
    'Irregular_Periods':  'descriptive',    # heterogeneous confound
    'Menopause_Status':   'descriptive',    # categorical, use binary version
    'Age_Menopause':      'descriptive',    # sparse (3.9%), surgical only
    'Reproductive_Span':  'descriptive',    # sparse (3.6%), surgical only

    # Domain 2 — Pregnancy history & APO
    'Ever_Pregnant':      'stratification', # defines APO analysis subgroup
    'Gravidity':          'descriptive',    # replaced by Pregnancy_Loss
    'Parity':             'descriptive',    # delivery burden — exposure variable, not a clustering input
    'Pregnancy_Loss':     'descriptive',    # replaces Gravidity — loss burden, not a clustering input
    'Nulliparous':        'descriptive',    # 6.3% of pregnant, small group
    'Has_GDM':            'descriptive',    # collinear with APO_Score
    'Has_Macrosomia':     'descriptive',    # collinear with APO_Score
    'APO_Score':          'descriptive',    # primary exposure — tested for cluster enrichment in notebook 07, not a clustering input

    # Domain 3 — Surgical history
    'Has_Hysterectomy':   'descriptive',    # sparse (3.8%) — below 5% threshold
    'Has_Oophorectomy':   'descriptive',    # very sparse (0.9%)
    'Surgical_Menopause': 'descriptive',    # sparse (3.9%) — below 5% threshold
    'Age_Oophorectomy':   'descriptive',    # very sparse (0.9%)

    # Domain 4 — Hormone therapy
    'Ever_HRT':           'descriptive',    # sparse (4.1%) — below 5% threshold
    'HRT_Type':           'descriptive',    # sparse, categorical
}

# ── Missingness and classification summary ─────────────────────────────────
print(f'\n{"═"*75}')
print(f'FEATURE MATRIX SUMMARY')
print(f'{"═"*75}\n')

print(f'  {"Feature":<22} {"Valid":>6} {"NaN":>6} {"% Valid":>8} '
      f'{"Use":>15}')
print(f'  {"─"*22} {"─"*6} {"─"*6} {"─"*8} {"─"*15}')

clustering_features     = []
descriptive_features    = []
stratification_features = []

for feat in expected_features:
    n_valid = features[feat].notna().sum()
    n_nan   = features[feat].isna().sum()
    pct     = n_valid / len(features) * 100
    use     = feature_classification[feat]

    print(f'  {feat:<22} {n_valid:>6,} {n_nan:>6,} {pct:>7.1f}% '
          f'{use:>15}')

    if use == 'clustering':
        clustering_features.append(feat)
    elif use == 'descriptive':
        descriptive_features.append(feat)
    else:
        stratification_features.append(feat)

print(f'\n{"─"*75}')
print(f'\nFeature counts by intended use:')
print(f'  Clustering:      {len(clustering_features):>3}  features → notebook 05')
print(f'  Descriptive:     {len(descriptive_features):>3}  features → notebook 04')
print(f'  Stratification:  {len(stratification_features):>3}  features → '
      f'subgroup definitions')

print(f'\nClustering features ({len(clustering_features)}):')
for f in clustering_features:
    n_valid = features[f].notna().sum()
    pct     = n_valid / len(features) * 100
    print(f'  {f:<22} {n_valid:>6,} valid  ({pct:.1f}%)')

# ── Add P_DEMO variables for notebook 03 ──────────────────────────────────
demo_cols = ['RIDAGEYR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR',
             'WTMECPRP', 'SDMVPSU', 'SDMVSTRA']

features_with_demo = features.copy()
for col in demo_cols:
    if col in df.columns:
        features_with_demo[col] = df[col]

print(f'\n✓ P_DEMO variables merged onto feature matrix')
print(f'  Final shape: {features_with_demo.shape}')
print(f'  ({len(expected_features)} reproductive features + '
      f'{len(demo_cols)} P_DEMO variables)')

# ── Validation ─────────────────────────────────────────────────────────────
assert features_with_demo.index.name == 'SEQN', \
    'SEQN is not the index'
assert len(features_with_demo) == 1603, \
    f'Row count changed: {len(features_with_demo)}'
assert all(f in features_with_demo.columns
           for f in expected_features), \
    'Some expected features missing from final matrix'
assert len(clustering_features) == 0, \
    (f'Expected 0 reproductive clustering features — clustering runs on '
     f'CKM biomarkers only (see notebook 05), got {len(clustering_features)}')

print(f'\n✓ All feature matrix assertions passed')

✓ All 19 expected features present
  Feature matrix shape: (1603, 19)

═══════════════════════════════════════════════════════════════════════════
FEATURE MATRIX SUMMARY
═══════════════════════════════════════════════════════════════════════════

  Feature                 Valid    NaN  % Valid             Use
  ────────────────────── ────── ────── ──────── ───────────────
  Age_Menarche            1,525     78    95.1%     descriptive
  Irregular_Periods       1,601      2    99.9%     descriptive
  Menopause_Status        1,602      1    99.9%     descriptive
  Age_Menopause              62  1,541     3.9%     descriptive
  Reproductive_Span          58  1,545     3.6%     descriptive
  Ever_Pregnant           1,601      2    99.9%  stratification
  Gravidity               1,114    489    69.5%     descriptive
  Parity                  1,135    468    70.8%     descriptive
  Pregnancy_Loss          1,100    503    68.6%     descriptive
  Nulliparous             1,601      2    99.9%  

---
## Section 9 · Export

The reproductive feature matrix is exported to the processed data 
directory for use in notebook 03. The export includes all 19 reproductive 
features plus 7 P_DEMO variables — 25 columns total — with SEQN as the 
index.

The feature classification documented in Section 8 is exported separately 
as a reference file documenting each reproductive feature's intended use 
(descriptive, exposure, or stratification). None of the reproductive 
features are used as clustering inputs in the current pipeline design — 
notebook 05 defines its clustering feature set independently from the 
CKM biomarker candidates established in notebook 03.

In [12]:
# ── Export reproductive feature matrix ────────────────────────────────────
OUTPUT_PATH_FEATURES = Path('../data/processed/reproductive_features.csv')
OUTPUT_PATH_CLASS    = Path('../data/processed/feature_classification.csv')

# ── File 1: Reproductive feature matrix ───────────────────────────────────
features_with_demo.to_csv(OUTPUT_PATH_FEATURES)

print(f'✓ Reproductive feature matrix exported')
print(f'  Path:  {OUTPUT_PATH_FEATURES}')
print(f'  Shape: {features_with_demo.shape}')
print(f'  Size:  {OUTPUT_PATH_FEATURES.stat().st_size / 1e3:.1f} KB')

# ── File 2: Feature classification reference ───────────────────────────────
classification_df = pd.DataFrame([
    {
        'feature':        feat,
        'intended_use':   feature_classification[feat],
        'n_valid':        features[feat].notna().sum(),
        'pct_valid':      round(features[feat].notna().sum() / len(features) * 100, 1),
        'clustering':     feature_classification[feat] == 'clustering',
    }
    for feat in expected_features
])

classification_df.to_csv(OUTPUT_PATH_CLASS, index=False)

print(f'\n✓ Feature classification exported')
print(f'  Path:  {OUTPUT_PATH_CLASS}')
print(f'  Shape: {classification_df.shape}')

# ── Validate exports ───────────────────────────────────────────────────────
assert OUTPUT_PATH_FEATURES.exists(), 'Feature matrix file not found'
assert OUTPUT_PATH_CLASS.exists(),    'Classification file not found'

# Verify feature matrix reads back correctly
verify = pd.read_csv(OUTPUT_PATH_FEATURES, index_col='SEQN')
assert verify.shape == features_with_demo.shape, \
    f'Readback shape mismatch: {verify.shape}'
assert verify.index.name == 'SEQN', \
    'SEQN not index in exported file'

print(f'\n✓ Export validation passed — file reads back correctly')
print(f'\n=== Notebook 02 outputs ===')
print(f'  reproductive_features.csv   — primary input for notebook 03')
print(f'  feature_classification.csv  — reproductive feature '
      f'intended-use reference')

✓ Reproductive feature matrix exported
  Path:  ../data/processed/reproductive_features.csv
  Shape: (1603, 26)
  Size:  190.9 KB

✓ Feature classification exported
  Path:  ../data/processed/feature_classification.csv
  Shape: (19, 5)

✓ Export validation passed — file reads back correctly

=== Notebook 02 outputs ===
  reproductive_features.csv   — primary input for notebook 03
  feature_classification.csv  — reproductive feature intended-use reference


---
## Section 10 · Summary

This notebook engineered 18 reproductive health features across four 
clinical domains from the analytical sample of 1,603 women aged 20–44. 
All features were derived from first principles using CDC codebook 
documentation, with engineering decisions explicitly justified on both 
clinical and methodological grounds.

### What this notebook established

**Domain 1 — Menstrual history (5 features):**
Age at menarche (mean 12.6 years), menstrual irregularity, menopause 
status, age at menopause, and reproductive span. A key discovery was 
that natural menopause cannot be identified in the 20–44 analytical 
sample — RHD043 code 7 (menopause) is absent in this age group. 
Menopause classification is therefore restricted to surgical menopause 
only (3.9%, n=63).

**Domain 2 — Pregnancy history & APO (8 features):**
Ever pregnant (71.7%), gravidity (mean 3.0), parity (mean 2.2), pregnancy
loss burden (mean 0.82), nulliparous flag (6.3% of pregnant), gestational
diabetes (11.7% of pregnant), macrosomia (12.5% of pregnant), and the 
composite APO_Score. The APO_Score distribution — 76.6% no APOs, 10.7% 
macrosomia only, 9.8% GDM only, 2.8% GDM + macrosomia — defines the 
exposure gradient for the CKM clustering analysis.

**Domain 3 — Surgical history (4 features):**
Hysterectomy (3.8%), oophorectomy (0.9%), surgical menopause composite 
(3.9%), and age at oophorectomy (descriptive only, n=15). All surgical 
features fall below the 5% clustering threshold and are retained for 
descriptive analysis only.

**Domain 4 — Hormone therapy (2 features):**
Ever used hormones (4.1%, n=66) and hormone therapy type (65.2% pills 
among users). Both features fall below the 5% clustering threshold. 
The more important role of hormone therapy in this pipeline is as a 
confounder for CKM biomarker interpretation — flagging will occur in 
notebook 03 via P_RXQ_RX linkage.

### Feature classification

| Use | N | Features |
|---|---|---|
| Clustering | 0 | None — clustering runs on CKM biomarkers only (notebook 05) |
| Descriptive | 17 | All reproductive features, including Age_Menarche, Pregnancy_Loss, Parity, and APO_Score |
| Stratification | 1 | Ever_Pregnant |

### Clustering design

None of the reproductive features engineered in this notebook are used 
as clustering inputs. CKM biomarker coverage analysis (completed in 
notebook 03) revealed that fasting lipids (triglycerides, LDL) are 
available for only about half of the analytical sample due to the 
NHANES fasting subsample design, so a two-stage approach is adopted:

- **Stage 1:** Cluster on non-fasting CKM biomarkers only (HbA1c, BMI, 
  mean systolic blood pressure, eGFR, HDL, glucose) — see notebook 05 
  for the exact feature set and exclusion criteria
- **Stage 2:** Characterize the resulting clusters using fasting lipids 
  in the fasting subsample, and test them for enrichment in APO history 
  (this notebook's exposure variables) in notebook 07

Reproductive history is deliberately excluded from Stage 1 to avoid 
circularity: APO_Score is the outcome this project ultimately tests 
against cluster membership, so it cannot also help define the clusters.

### Known limitations

- Hypertensive disorders of pregnancy absent from this release — 
  APO_Score captures metabolic APOs only/
- APO_Score weighting is a simplification — the 2:1 GDM-to-macrosomia\weighting
  reflects relative clinical severity, not a validated clinical scoring system.
- Irregular_Periods confounded by contraceptive use — unresolvable 
  without P_RXQ_RX linkage.
- Natural menopause not identifiable in 20–44 sample.
- Gravidity, Parity, and APO_Score missing for never-pregnant women 
  (structural — not imputable).
- Fasting lipids available for 50.2% of sample only.


### Output files

| File | Shape | Description |
|---|---|---|
| `reproductive_features.csv` | 1,603 × 25 | Primary input for notebook 03 |
| `feature_classification.csv` | 18 × 5 | Reproductive feature intended-use reference (descriptive documentation only — not read by notebook 05) |

### What notebook 03 will do

Notebook 03 loads `reproductive_features.csv` and links it to five CKM 
biomarker modules via SEQN: P_GHB (HbA1c), P_BMX (anthropometrics), 
P_BPXO (blood pressure), P_BIOPRO (biochemistry), and P_HDL (HDL 
cholesterol). It will also link P_RXQ_RX (prescription medications) 
to flag women on CKM-relevant medications — metformin, antihypertensi